# Predicción horaria con tres torres semánticas GRU-D

Este notebook entrena tres torres multivariables con atención temporal. Sus resúmenes se combinan con el contexto del paciente para predecir la sepsis por hora.

La configuración inicial usa todos los pacientes del conjunto A de entrenamiento y validación cruzada de tres folds. La prueba final está desactivada por defecto.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.

In [2]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.models.semantic_grud import SEMANTIC_GROUPS_3
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATA_DIR = ROOT / "data" / "processed"


## 1. Configuración del piloto

El piloto usa 2.000 pacientes y una sola partición de validación. `source` admite `"A"`, `"B"` y `"A+B"`.

La búsqueda usa Optuna TPE con 10 ensayos iniciales, incluido el modelo base. La poda por mediana puede actuar desde el ensayo 6, al final de la época 8, con al menos 3 resultados de referencia en esa época. Para validar una configuración fija con 3 folds, usa `search="random"`, `trials=1`, `sample_patients=None` y `folds=3`.


In [11]:
settings = dict(
    architecture="semantic_grud_3",
    source="A",
    sample_patients=2000,
    validation_fraction=0.20,
    folds=None,
    search="tpe",
    trials=20,
    epochs=30,
    patience=4,
    seed=2026,
)

base = Hyperparameters(
    window=12,
    units=8,
    dense_units=32,
    dropout=0.20,
    learning_rate=0.001,
    batch_size=128,
)


## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [12]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(split_patients(
    frame,
    validation_fraction=settings["validation_fraction"],
    folds=settings["folds"],
    seed=settings["seed"],
    sample_patients=settings["sample_patients"],
))

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append({
        "partición": name,
        "pacientes": len(patient_labels),
        "pacientes positivos": int(patient_labels.sum()),
        "horas": len(partition),
    })
display(pd.DataFrame(summary).set_index("partición"))


,pacientes,pacientes positivos,horas
partición,,,
train,1600,141,62270
validation,400,35,15554


In [13]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print("Horas válidas de la primera ventana:", inputs["valid_hours"][0].astype(int).tolist())
print("Etiqueta de la primera ventana:", int(labels[0, 0]))

for tower, variables in SEMANTIC_GROUPS_3.items():
    print(tower, list(variables))


Formas de entrada: {'temporal': (11, 15, 26, 4), 'valid_hours': (11, 15), 'static': (11, 10)}
Forma de las etiquetas: (11, 1)
Paciente: A_015805
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0
hemodynamics_oxygenation ['HR', 'SBP', 'MAP', 'DBP', 'Lactate', 'Resp', 'O2Sat', 'SaO2', 'FiO2', 'PaCO2', 'pH', 'Temp']
renal_hepatic ['BUN', 'Creatinine', 'Calcium', 'Magnesium', 'Phosphate', 'Potassium', 'Glucose', 'AST', 'Alkalinephos', 'Bilirubin_total']
hematology ['Hct', 'Hgb', 'WBC', 'Platelets']


## 3. Entrenar y revisar los resultados

Cada ejecución crea una carpeta nueva en `runs/` con el modelo, la configuración, las particiones, las métricas y los tiempos. El prefijo `notebook-semantic-grud-3` identifica estas corridas.

Para comparar con `modeling_semantic.ipynb` (cinco torres), comprueba que las asignaciones de pacientes en `splits.csv` coincidan y usa el mismo origen de prueba. Una vez elegida la configuración de cada modelo, compara la precisión promedio (AP), la utilidad y el recall al umbral seleccionado en los archivos `test_<origen>_metrics.csv`.

In [14]:
run_dir = ROOT / "runs" / f"notebook-semantic-grud-3-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))


Trial 1/10, fold 1/1: Hyperparameters(window=15, units=8, dense_units=32, dropout=0.2, learning_rate=0.001, batch_size=128)
Epoch 1/40
487/487 - 48s - 99ms/step - loss: 0.1191 - pr_auc: 0.0262 - val_loss: 0.1028 - val_pr_auc: 0.0366
Epoch 2/40
487/487 - 32s - 65ms/step - loss: 0.0904 - pr_auc: 0.1102 - val_loss: 0.0993 - val_pr_auc: 0.0595
Epoch 3/40
487/487 - 27s - 55ms/step - loss: 0.0824 - pr_auc: 0.1982 - val_loss: 0.0991 - val_pr_auc: 0.0698
Epoch 4/40
487/487 - 41s - 83ms/step - loss: 0.0769 - pr_auc: 0.2594 - val_loss: 0.1013 - val_pr_auc: 0.0715
Epoch 5/40
487/487 - 26s - 54ms/step - loss: 0.0721 - pr_auc: 0.3133 - val_loss: 0.1060 - val_pr_auc: 0.0709
Epoch 6/40
487/487 - 56s - 116ms/step - loss: 0.0679 - pr_auc: 0.3674 - val_loss: 0.1035 - val_pr_auc: 0.0813
Epoch 7/40
487/487 - 42s - 87ms/step - loss: 0.0642 - pr_auc: 0.4128 - val_loss: 0.1063 - val_pr_auc: 0.0840
Epoch 8/40
487/487 - 37s - 75ms/step - loss: 0.0605 - pr_auc: 0.4581 - val_loss: 0.1091 - val_pr_auc: 0.0891
Epo

: 

In [ ]:
display(pd.read_csv(run_dir / "trials.csv").round(4))

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial", "fold", "train_windows", "validation_windows",
    "preparation_seconds", "epochs_run", "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


run_timing = json.loads((run_dir / "run_timing.json").read_text(encoding="utf-8"))
print(f"Tiempo total de ejecución: {run_timing['total_seconds']:.2f} s")

## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [ ]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(run_dir / f"test_{test_source}_predictions.csv", index=False)
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))
